[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vtavakkoli/TinyCeNN-LM/blob/main/notebooks/Jev_CeNN_All28_FP16_Stable_Joint_Substitution_V7_Colab.ipynb)

# Jev-CeNN All-28 V7 — Stable FP16 + Mergeable Attention Training

**V7.4 fix:** Resolve Float/Half mismatches in auxiliary action-head inference by pinning the corrected standalone runtime and wrapping end-to-end FP16 benchmarking/reload parity in an explicit FP16 autocast context. Both report and complete model ZIP downloads remain enabled. Already-trained in-memory models can proceed from the benchmarking cell without repeating training.

**V7.3 fix:** Replace invalid FP16 fingerprint inequality assertion with an exact per-tensor teacher FP16 comparison. Export two downloadable ZIP archives: a lightweight results package and a full FP16 standalone-model package.

**V7.2 CUDA fix:** The training cell automatically moves the student model and buffers onto the active indexed CUDA device before LoRA installation. A previous strict `torch.device("cuda") == torch.device("cuda:0")` assertion incorrectly failed for healthy GPU tensors. Restart and run all cells to refresh the setup.


**Goal:** New full model with all 28 ATTENTION submodules replaced, original FFNs and attention/FFN normalizations untouched, reduced FP16 checkpoint size, and higher decision fidelity than unstable V6.

### Evidence-driven fixes
- V6 reported **`initialized_from: partial_or_random_initialized`**. V7 searches for a full V4 checkpoint ZIP or V5/V6 Hugging Face model first. If none are available, V7 initializes the replacement QKV/WO directly from the ORIGINAL pretrained model and uses **native-equivalent attention masks** (full global / W129 local) instead of random or untrained sparse masks. This is a valid new all-28 substitution, but does NOT reuse your earlier trained V4 attention weights.
- V6 DEV teacher agreement fell **94.17% → 93.33%** and JS divergence worsened; V7 **retains the initial model and the best DEV checkpoint** rather than always saving the final SGD update.
- V6's CeNN learning rate **0.01→0.002** was applied to every CeNN filter, with unstable accuracy and slow inference. V7 applies this schedule to **bounded CeNN gains** only; core CeNN filters train at 1/100th rate, and rank-4/8 **LoRA on all 28 QKV/output projections** trains at 1/100th rate. LoRA weights are **merged before export**, so no adapter kernel adds latency.
- Old V6 long-sequence speedup ratio was **0.696×**; V7 preserves the 28 converted attentions but automatically tests optional CeNN-off inference per layer to reduce overhead without changing FFNs.
- `summary_v2(1).json` describes a **different ternary packing test** and is NOT used as a compatible checkpoint. V7 is **FP16, not 2-bit ternary**.

### Model output
The exported `model.safetensors` contains ALL 28 substituted attention layers, untouched pretrained FFNs/norms and original task head in **actual FP16 storage**. It also includes tokenizer, config, per-layer CeNN status, Hugging Face loader, full training history, DEV/holdout accuracy and paired FP16 latency. The loader builds an FP16 model *before* reading weights to avoid accidentally upcasting to FP32. Optional private-by-default Hugging Face upload.

**Important:** The fallback has no guaranteed GPU speedup: wide/full attention requires dense computation. Full-model fidelity is measured rather than guaranteed; training checkpoint selection has no minimum 99% quality gate, and all 28 attention replacements remain installed. CUDA GPU recommended (T4/L4/A100). Original source: `convaiinnovations/laya-typed-decisions`; `vtava/Laya-1.2-speedup-standalone-fp16` is NOT an interchangeable model without a validated conversion.


**V7.1 CUDA LoRA fix:** LoRA A/B factors now inherit their base projection's CUDA device and dtype, with an optimizer-placement assertion, training-forward smoke test and safe Cell-5 reruns after an earlier device mismatch. The 28 attention substitutions and learning-rate schedule are unchanged.


In [1]:
#@title 1. Environment + accuracy-preserving FP16 training configuration
%pip -q install "transformers==4.57.6" "datasets>=2.20,<5" "huggingface_hub>=0.25,<1" "safetensors>=0.4"
%pip -q install -U "git+https://github.com/vtavakkoli/TinyCeNN-LM.git@645723441fb81c62d59231bffd29a177a5edb3ac"

from pathlib import Path
from datetime import datetime, timezone
import copy, csv, gc, hashlib, json, math, os, random, shutil, time, zipfile
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
from datasets import load_dataset
from transformers import AutoTokenizer
from safetensors.torch import save_model
from huggingface_hub import HfApi, snapshot_download, get_token, login
from tinycenn_lm.standalone_decision import (
    QTYPES, build_checkpoint_model, build_sequence, collate_items,
    normalize_question, full_attention_indices
)
from tinycenn_lm.all28_fp16 import WindowCeNNAttention, load_all28, build_all28_from_dir
import tinycenn_lm.all28_fp16 as fp16_loader

SEED=2028
SOURCE_MODEL="convaiinnovations/laya-typed-decisions"
INITIAL_HF_REPO="" #@param {type:"string"}
V4_ZIP_PATH="/content/all28_results.zip" #@param {type:"string"}
V4_DIR="/content/Jev_CeNN_All28_AttentionOnly_BruteForce_V4" #@param {type:"string"}
# Trained checkpoints have priority. When none exist, use original pretrained
# QKV/WO and native-equivalent attention instead of randomly initialized masks.
ALLOW_NATIVE_EQUIVALENT_FALLBACK=True #@param {type:"boolean"}
FALLBACK_GLOBAL_WINDOW=512  # native global full-attention equivalent
FALLBACK_LOCAL_WINDOW=129   # native sliding-attention equivalent
FALLBACK_CENN_ENABLED=True  # zero-gain CeNN branch trains from identity
LR_GAIN_START=0.01 #@param {type:"number"}
LR_GAIN_END=0.002 #@param {type:"number"}
LORA_LR_MULT=0.01
CENN_CORE_LR_MULT=0.01
LORA_RANK=4 #@param {type:"integer"}
LORA_ALPHA=4.0
JOINT_STEPS=250 #@param {type:"integer"}
BATCH=2 #@param {type:"integer"}
GRAD_ACCUM=4 #@param {type:"integer"}
EVAL_EVERY=25 #@param {type:"integer"}
MAX_GRAD_NORM=0.35
KL_WEIGHT=2.0
GOLD_WEIGHT=0.03
KD_TEMPERATURE=2.0
TRAIN_CENN_CORE=True #@param {type:"boolean"}
TRAIN_LORA_ALL_28=True #@param {type:"boolean"}
# Preserve V4's winning CeNN on/off choices by default instead of
# reactivating 28 unnecessary conv branches as V6 did.
ENABLE_CENN_ALL=False #@param {type:"boolean"}
AUTOPRUNE_CENN=True #@param {type:"boolean"}
PRUNE_DECISIONS=64
MAX_CENN_PRUNES=28
GRAD_CHECKPOINTING=True
FP16_AUTOCAST=True
TRAIN_CASES=700
DEV_CASES=220
HOLDOUT_CASES=200
PROBE_CASES=80
PROBE_BATCHES=12
DEV_GATE_DECISIONS=240
HOLDOUT_DECISIONS=500
TRAIN_DECISIONS_LIMIT=1200
MAX_LEN=512
BENCH_REPEATS=12
# Download results to this browser after local export and reload verification.
AUTO_DOWNLOAD_RESULTS_ZIP=True #@param {type:"boolean"}
AUTO_DOWNLOAD_MODEL_ZIP=True #@param {type:"boolean"}
UPLOAD_TO_HF=True #@param {type:"boolean"}
HF_PRIVATE=True #@param {type:"boolean"}
HF_REPO_NAME="jev-cenn-all28-fp16-stable-v7" #@param {type:"string"}
OUTPUT_DIR=Path("/content/Jev_CeNN_All28_FP16_Stable_V7")
EXPORT_DIR=OUTPUT_DIR/"huggingface_model"
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
EXPORT_DIR.mkdir(parents=True,exist_ok=True)
assert (LR_GAIN_START,LR_GAIN_END)==(0.01,0.002)
assert JOINT_STEPS>0 and LORA_RANK>0 and GRAD_ACCUM>0
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
if not torch.cuda.is_available():raise RuntimeError("Please enable CUDA GPU in Google Colab.")
torch.cuda.manual_seed_all(SEED)
# Canonical indexed CUDA device: torch.device("cuda") != torch.device("cuda:0")
# under device equality, even though the former defaults to device 0.
device=torch.device("cuda",torch.cuda.current_device())
amp_dtype=torch.float16  # Requested genuine FP16 training computation
scaler=torch.amp.GradScaler("cuda",enabled=FP16_AUTOCAST)
source_dir=Path(snapshot_download(
    SOURCE_MODEL,allow_patterns=[
        "rl_agent_config.json","model.safetensors","encoder/*","tokenizer/*"]))
teacher,source_cfg=build_checkpoint_model(source_dir)
student_eval,_=build_checkpoint_model(source_dir)
teacher.eval().requires_grad_(False)
student_eval.eval().requires_grad_(False)
teacher.encoder.config.reference_compile=False
student_eval.encoder.config.reference_compile=False
tokenizer=AutoTokenizer.from_pretrained(str(source_dir/"tokenizer"))
GLOBAL_LAYERS=tuple(full_attention_indices(teacher))
assert len(teacher.encoder.layers)==28
assert tuple(GLOBAL_LAYERS)==tuple(range(0,28,3))
print("GPU",torch.cuda.get_device_name(0),"FP16 autocast",
      FP16_AUTOCAST,"gain LR",LR_GAIN_START,"→",LR_GAIN_END)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 55.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 62.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.29.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

tokenizer.json: 0.00B [00:00, ?B/s]

rl_agent_config.json:   0%|          | 0.00/847 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/337 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/843M [00:00<?, ?B/s]

GPU Tesla T4 FP16 autocast True gain LR 0.01 → 0.002


In [2]:
#@title 3. Partition dataset by CASE: TRAIN, PROBE, DEV-GATE and sealed HOLDOUT
def _jsonish(x):
    if isinstance(x, str):
        try:
            return json.loads(x)
        except Exception:
            return x
    return x

def _canon_label(x):
    s = str(x).strip().lower()
    try:
        v = float(s)
        if math.isfinite(v) and v.is_integer():
            return str(int(v))
    except Exception:
        pass
    return s

def _gold_metadata(q, gold_answer):
    if not isinstance(gold_answer, dict):
        raise ValueError("missing gold answer")
    probs_raw = _jsonish(gold_answer.get("probabilities", {}))
    if not isinstance(probs_raw, dict) or len(probs_raw) < 2:
        raise ValueError("gold probabilities missing or malformed")

    gold_keys = [str(k) for k in probs_raw.keys()]
    if q["t"] == "choice":
        labels = [str(k) for k in q["crit"].keys()]
    elif q["t"] == "noul":
        labels = ["false", "true"]
    else:
        labels = list(gold_keys)
        if all(_canon_label(k).lstrip("-").isdigit() for k in labels):
            labels = sorted(labels, key=lambda k: int(_canon_label(k)))

    lookup = {_canon_label(k): float(v) for k, v in probs_raw.items()}
    if len(labels) != len(gold_keys) or any(_canon_label(k) not in lookup for k in labels):
        raise ValueError("Gold labels do not match encoded option order")

    gold_probs = np.asarray([lookup[_canon_label(k)] for k in labels], dtype=np.float64)
    gold_probs = np.clip(gold_probs, 0.0, None)
    if not np.isfinite(gold_probs).all() or gold_probs.sum() <= 0:
        raise ValueError("invalid gold probability vector")
    gold_probs /= gold_probs.sum()

    gold_label = _canon_label(gold_answer.get("label", labels[int(gold_probs.argmax())]))
    canon_labels = [_canon_label(k) for k in labels]
    if gold_label not in canon_labels:
        raise ValueError(f"gold label {gold_label!r} not in {labels!r}")
    return labels, gold_probs.tolist(), canon_labels.index(gold_label)

def make_items_from_rows(rows, seed, shuffle_items):
    out = []
    head_max_len = int(source_cfg.get("head_max_len", 192))
    for row in rows:
        state = _jsonish(row["state"])
        questions = _jsonish(row["questions"])
        gold = _jsonish(row["gold"])
        case_id = str(row.get("id", ""))
        workflow = str(row.get("workflow", "unknown"))
        state_text = state if isinstance(state, str) else json.dumps(state, ensure_ascii=False)
        state_ids = tokenizer(
            state_text.replace(tokenizer.mask_token, " "),
            add_special_tokens=False,
        )["input_ids"]

        for qid, qdef in questions.items():
            q = normalize_question(qdef)
            ids, markers = build_sequence(
                tokenizer,
                state,
                q,
                max_len=min(MAX_LEN, int(source_cfg.get("max_len", MAX_LEN))),
                head_max_len=head_max_len,
                truncate_left=isinstance(state, list),
                state_ids=state_ids,
            )
            if len(markers) < 2:
                continue
            labels, gold_probs, gold_index = _gold_metadata(q, gold[qid])
            if len(labels) != len(markers):
                raise ValueError(
                    f"{case_id}/{qid}: {len(markers)} markers vs {len(labels)} gold labels"
                )
            out.append({
                "ids": ids,
                "markers": markers,
                "qtype": QTYPES[q["t"]],
                "qtype_name": q["t"],
                "qid": str(qid),
                "case_id": case_id,
                "workflow": workflow,
                "labels": labels,
                "gold_probs": gold_probs,
                "gold_index": int(gold_index),
            })
    if shuffle_items:
        random.Random(seed).shuffle(out)
    return out

from tinycenn_lm.jev_cenn_selective import (
    verify_case_partition, judge_replacement, plan_active_layers,
)
rows=list(load_dataset("LocalLLaMA/typed-decisions","all",split="train"))
rng=random.Random(SEED)
rng.shuffle(rows)
assert TRAIN_CASES+DEV_CASES+HOLDOUT_CASES<=len(rows), (
    f"Required TRAIN+DEV+HOLDOUT cases exceeds {len(rows)} dataset TRAIN cases")
train_rows=rows[:TRAIN_CASES]
dev_rows=rows[TRAIN_CASES:TRAIN_CASES+DEV_CASES]
holdout_rows=rows[TRAIN_CASES+DEV_CASES:TRAIN_CASES+DEV_CASES+HOLDOUT_CASES]
split_info=verify_case_partition(train_rows,dev_rows,holdout_rows)
probe_rows=dev_rows[:PROBE_CASES]
gate_rows=dev_rows[PROBE_CASES:]
verify_case_partition(train_rows,probe_rows,gate_rows)
assert not {str(x["id"]) for x in holdout_rows}&{
    str(x["id"]) for x in dev_rows}
train_items=make_items_from_rows(train_rows,SEED,True)
probe_items=make_items_from_rows(probe_rows,SEED+1,True)
gate_items=make_items_from_rows(gate_rows,SEED+2,True)
# IMPORTANT: holdout_items are NOT constructed yet. The final evaluation
# cell materializes these previously sealed rows after all decisions are made.
assert len(gate_items)>=DEV_GATE_DECISIONS
assert len(probe_items)>=PROBE_BATCHES*BATCH
print("Independent partitions:",split_info,
      "probe cases",len(probe_rows),"gate cases",len(gate_rows))
print("Optimizer/probe/gate decisions:",len(train_items),
      len(probe_items),len(gate_items))

def direct_batch(selected):
    b = collate_items(selected, int(tokenizer.pad_token_id))
    return {k: v.to(device) for k, v in b.items()}

def batch(items, offset, n=BATCH):
    selected = [items[(offset + i) % len(items)] for i in range(n)]
    return direct_batch(selected)

# Only case-isolated TRAIN / PROBE / DEV-GATE inputs are available here.
# The independent HOLDOUT remains sealed until the final evaluation cell.
print("TRAIN / PROBE / DEV-GATE decisions:",
      len(train_items), len(probe_items), len(gate_items))
print("Official TEST untouched; independent HOLDOUT not materialized.")


README.md: 0.00B [00:00, ?B/s]

all/train-00000-of-00001.parquet:   0%|          | 0.00/599k [00:00<?, ?B/s]

all/test-00000-of-00001.parquet:   0%|          | 0.00/222k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1200 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/400 [00:00<?, ? examples/s]

Independent partitions: {'train_cases': 700, 'selection_cases': 220, 'holdout_cases': 200, 'all_partitions_disjoint': True} probe cases 80 gate cases 140
Optimizer/probe/gate decisions: 3500 400 700
TRAIN / PROBE / DEV-GATE decisions: 3500 400 700
Official TEST untouched; independent HOLDOUT not materialized.


In [3]:
#@title 3. Complete trained initialization — never silently use random substitutions
def ffn_norm_digest(model):
    sha=hashlib.sha256()
    for i,l in enumerate(model.encoder.layers):
        for m in ("mlp","mlp_norm","attn_norm"):
            for k,t in getattr(l,m).state_dict().items():
                sha.update(f"{i}/{m}/{k}".encode())
                sha.update(t.detach().float().cpu().contiguous().numpy().tobytes())
    return sha.hexdigest()

BASELINE_FFN_HASH=ffn_norm_digest(student_eval)
manifest_layers=[]
init_method=None
if INITIAL_HF_REPO.strip():
    loaded=load_all28(INITIAL_HF_REPO.strip(),device="cpu")
    student_eval=loaded.model.cpu().float().eval().requires_grad_(False)
    del loaded
    init_method="huggingface_full_28"
    if ffn_norm_digest(student_eval)!=BASELINE_FFN_HASH:
        raise RuntimeError("Initialization changed original FFN/norm tensors. "
                           "A separately FP16-rounded source needs a matching "
                           "reference hash; refusing to train against wrong FFNs.")
    for i,layer in enumerate(student_eval.encoder.layers):
        a=layer.attn
        assert isinstance(a,WindowCeNNAttention)
        manifest_layers.append({
          "layer":i,"window":a.window,"pattern":a.pattern,
          "anchors":a.anchors,"cenn_on":bool(a.use_cenn)})
else:
    files={}
    for i in range(28):
        name=f"forced_best_attention_L{i:02d}.pt"
        for folder in (Path(V4_DIR),Path("/content"),
                       Path("/content/Jev_CeNN_All28_HF_Export_V5/per_layer_checkpoints")):
            found=folder/name
            if found.is_file():
                files[i]=found
                break
    zip_path=Path(V4_ZIP_PATH)
    if zip_path.is_file():
        imported=OUTPUT_DIR/"v4_import"
        imported.mkdir(exist_ok=True)
        with zipfile.ZipFile(zip_path) as z:
            available={Path(m).name:m for m in z.namelist()
                       if Path(m).name.startswith("forced_best_attention_L")
                       and m.endswith(".pt")}
            for i in range(28):
                if i in files:continue
                name=f"forced_best_attention_L{i:02d}.pt"
                if name in available:
                    p=imported/name
                    p.write_bytes(z.read(available[name]))
                    files[i]=p
    missing=sorted(set(range(28))-set(files))
    if missing and not ALLOW_NATIVE_EQUIVALENT_FALLBACK:
        raise RuntimeError(
            f"Missing V4 checkpoints for layers {missing}. "
            "Upload the V4 all28_results.zip, set INITIAL_HF_REPO, or "
            "enable ALLOW_NATIVE_EQUIVALENT_FALLBACK.")
    if missing:
        print(
            f"Found {len(files)}/28 V4 attention checkpoints. "
            f"Initializing the remaining {len(missing)} attentions from "
            "ORIGINAL pretrained weights with native-equivalent masks. "
            "This does NOT reproduce the earlier V4 trained checkpoints.")
    init_method=(
        "v4_all_28_saved_attention_layers" if not missing
        else "original_pretrained_native_equivalent_all28"
             if not files else f"v4_partial_{len(files)}_plus_native_equivalent"
    )
    for i in range(28):
        original=student_eval.encoder.layers[i].attn
        if i in files:
            obj=torch.load(files[i],map_location="cpu",weights_only=True)
            assert int(obj["layer"])==i
            config=obj["config"]
            assert len(config)==3
            w,pat,anchors=config
            a=WindowCeNNAttention(
                original,int(w),str(pat),int(anchors))
            a.load_state_dict(obj["state"],strict=True)
            a.use_cenn=bool(obj["cenn_on"])
        else:
            # Every substituted projection/rotary component is cloned from
            # the ORIGINAL pretrained attention. No random QKV replacement.
            # CeNN's initial correction gain is EXACTLY ZERO.
            if i in GLOBAL_LAYERS:
                w,pat,anchors=FALLBACK_GLOBAL_WINDOW,"full",0
            else:
                w,pat,anchors=FALLBACK_LOCAL_WINDOW,"local",0
            a=WindowCeNNAttention(original,int(w),str(pat),int(anchors))
            a.use_cenn=bool(FALLBACK_CENN_ENABLED)
            assert float(a.cenn_gain.detach())==0.0
        student_eval.encoder.layers[i].attn=a
        manifest_layers.append({
            "layer":i,"window":int(w),"pattern":str(pat),
            "anchors":int(anchors),"cenn_on":bool(a.use_cenn)})
assert len(manifest_layers)==28
assert all(isinstance(layer.attn,WindowCeNNAttention)
           for layer in student_eval.encoder.layers)
if ENABLE_CENN_ALL:
    for layer in student_eval.encoder.layers:layer.attn.use_cenn=True
    for p in manifest_layers:p["cenn_on"]=True
assert ffn_norm_digest(student_eval)==BASELINE_FFN_HASH
# Guarantee all 28 ATTENTION modules are substituted and original FFNs frozen.
assert len(manifest_layers)==28
assert set(x["layer"] for x in manifest_layers)==set(range(28))
student_eval.eval().requires_grad_(False)
print("Initialization:",init_method,
      "CeNN enabled layers:",[x["layer"] for x in manifest_layers if x["cenn_on"]])
print("Native FFNs + norms SHA256:",BASELINE_FFN_HASH)


Found 0/28 V4 attention checkpoints. Initializing the remaining 28 attentions from ORIGINAL pretrained weights with native-equivalent masks. This does NOT reproduce the earlier V4 trained checkpoints.
Initialization: original_pretrained_native_equivalent_all28 CeNN enabled layers: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27]
Native FFNs + norms SHA256: 766166b2aa6376e9023fff6e2b63a2cb80618f2065597342f5cafe8be57670e6


In [4]:
#@title 4. Length-bucketed teacher cache (teacher forwarded once)
# Independently partitioned DEV is used for checkpoint ranking only.
teacher.to(device).eval().requires_grad_(False)
student_eval.to(device).eval().requires_grad_(False)
def bucket_records(items,batch=BATCH,shuffle=False):
    ordered=sorted(items,key=lambda x:len(x["ids"]))
    # Equal-ish lengths reduce padding in the frozen teacher cache.
    chunks=[ordered[i:i+batch] for i in range(0,len(ordered),batch)]
    if shuffle:random.Random(SEED+511).shuffle(chunks)
    records=[]
    for chunk in chunks:
        inp=direct_batch(chunk)
        with torch.inference_mode(),torch.autocast("cuda",dtype=amp_dtype,
                                                     enabled=FP16_AUTOCAST):
            logits,_=teacher(**inp)
        records.append({
          "inputs":{k:v.detach().cpu().contiguous() for k,v in inp.items()},
          "teacher":logits.float().detach().cpu().contiguous(),
          "gold":torch.tensor([r["gold_index"] for r in chunk],dtype=torch.long),
          "length":int(inp["input_ids"].shape[1])})
    return records

TRAIN_ITEMS=train_items[:min(TRAIN_DECISIONS_LIMIT,len(train_items))]
GATE_ITEMS=gate_items[:DEV_GATE_DECISIONS]
assert len(GATE_ITEMS)==DEV_GATE_DECISIONS
train_records=bucket_records(TRAIN_ITEMS,shuffle=True)
gate_records=bucket_records(GATE_ITEMS)
teacher.cpu()
gc.collect();torch.cuda.empty_cache()
assert ffn_norm_digest(student_eval)==BASELINE_FFN_HASH
@torch.inference_mode()
def eval_quality(model,records,max_decisions=None):
    model.eval()
    agree=gold_hits=teacher_gold=n=0
    js_total=0.
    for rec in records:
        inp={k:v.to(device,non_blocking=True) for k,v in rec["inputs"].items()}
        with torch.autocast("cuda",dtype=amp_dtype,enabled=FP16_AUTOCAST):
            p,_=model(**inp)
        valid=inp["marker_mask"]
        student_logits=p.float().masked_fill(~valid,-1e4)
        teacher_logits=rec["teacher"].to(device).masked_fill(~valid,-1e4)
        a=F.softmax(teacher_logits,dim=-1)
        b=F.softmax(student_logits,dim=-1)
        reference=a.argmax(-1);student=b.argmax(-1)
        truths=rec["gold"].to(device)
        agree+=int((reference==student).sum())
        gold_hits+=int((student==truths).sum())
        teacher_gold+=int((reference==truths).sum())
        mid=0.5*(a+b)
        js=0.5*((a*(a.clamp_min(1e-9).log()-
                         mid.clamp_min(1e-9).log())).sum(-1)
                 +(b*(b.clamp_min(1e-9).log()-
                         mid.clamp_min(1e-9).log())).sum(-1))
        js_total+=float(js.sum())
        n+=int(student.shape[0])
        if max_decisions is not None and n>=max_decisions:break
    return {"n":n,"teacher_agreement":agree/n,
            "gold_accuracy":gold_hits/n,
            "teacher_gold_accuracy":teacher_gold/n,
            "mean_js":js_total/n}

BASELINE_DEV=eval_quality(student_eval,gate_records)
print("Initialization DEV (FP16 compute):",BASELINE_DEV)
print("Cached",len(train_records),"train batches,",
      len(gate_records),"DEV batches. Teacher released from GPU.")


Initialization DEV (FP16 compute): {'n': 240, 'teacher_agreement': 0.9916666666666667, 'gold_accuracy': 0.8333333333333334, 'teacher_gold_accuracy': 0.8416666666666667, 'mean_js': 0.00010349645356958112}
Cached 600 train batches, 120 DEV batches. Teacher released from GPU.


In [5]:
#@title 5. LoRA all 28 attentions (mergeable) + bounded CeNN-gain training
class MergeableLoRA(nn.Module):
    # Strictly training-time, never shipped to Hub.
    def __init__(self,linear,rank=LORA_RANK,alpha=LORA_ALPHA):
        super().__init__()
        if not isinstance(linear,nn.Linear):
            raise TypeError("Expected nn.Linear projection for LoRA")
        self.base=linear
        self.base.requires_grad_(False)
        # V7.1: Build parameters on the original projection's CUDA device
        # and dtype. A default torch.empty()/torch.zeros() lives on CPU
        # even if student_eval was already moved to CUDA in cell 4.
        self.lora_A=nn.Parameter(
            linear.weight.new_empty((rank,linear.in_features)))
        self.lora_B=nn.Parameter(
            linear.weight.new_zeros((linear.out_features,rank)))
        nn.init.normal_(self.lora_A,mean=0,std=0.01)
        assert self.lora_A.device==self.base.weight.device
        assert self.lora_B.device==self.base.weight.device
        assert self.lora_A.dtype==self.base.weight.dtype
        assert self.lora_B.dtype==self.base.weight.dtype
        self.scale=float(alpha)/rank
    def forward(self,x):
        return self.base(x)+F.linear(F.linear(x,self.lora_A),self.lora_B)*self.scale
    @torch.no_grad()
    def merge(self):
        self.base.weight.add_((self.lora_B@self.lora_A)*self.scale)
        return self.base

lora_params=[];gain_params=[];core_params=[]
# Robust to rerunning Cell 5, to a prior interrupted LoRA install, and
# to the PyTorch cuda versus cuda:0 device comparison distinction.
# Strip any incomplete training-only wrappers BEFORE moving the model.
for layer in student_eval.encoder.layers:
    a=layer.attn
    if isinstance(a.Wqkv,MergeableLoRA):
        a.Wqkv=a.Wqkv.base
    if isinstance(a.Wo,MergeableLoRA):
        a.Wo=a.Wo.base
# Explicitly place ALL pretrained weights, FFNs, CeNN branches, and buffers
# on this runtime's CUDA device. This is idempotent if already on CUDA.
device=torch.device("cuda",torch.cuda.current_device())
student_eval.to(device).eval()
wrong_locations=[(name,str(p.device))
                 for name,p in student_eval.named_parameters()
                 if p.device.type!="cuda" or p.device.index!=device.index]
wrong_buffers=[(name,str(b.device))
               for name,b in student_eval.named_buffers()
               if b.device.type!="cuda" or b.device.index!=device.index]
if wrong_locations or wrong_buffers:
    raise RuntimeError(
        f"Model placement failed on {device}: "
        f"params={wrong_locations[:5]}, buffers={wrong_buffers[:5]}")
print(f"CUDA placement verified: {len(list(student_eval.parameters()))} "
      f"parameter tensors on {device}")
for layer in student_eval.encoder.layers:
    a=layer.attn
    a.requires_grad_(False)
    if TRAIN_LORA_ALL_28:
        a.Wqkv=MergeableLoRA(a.Wqkv)
        a.Wo=MergeableLoRA(a.Wo)
        lora_params.extend([a.Wqkv.lora_A,a.Wqkv.lora_B,
                            a.Wo.lora_A,a.Wo.lora_B])
    if a.use_cenn:
        a.cenn_gain.requires_grad_(True)
        gain_params.append(a.cenn_gain)
        if TRAIN_CENN_CORE:
            a.cenn.requires_grad_(True)
            core_params+=list(a.cenn.parameters())
assert len(lora_params)+len(gain_params)+len(core_params)>0
# Check each trainable tensor before creating the optimizer and running 250 steps.
# This catches LoRA, CeNN-gain, and core tensors left on CPU early.
trainables=lora_params+gain_params+core_params
bad_devices=[
    (kind,index,str(p.device),str(p.dtype))
    for kind,group in (("LoRA",lora_params),("CeNN gain",gain_params),
                       ("CeNN core",core_params))
    for index,p in enumerate(group)
    if p.device.type!="cuda" or p.device.index!=device.index
]
if bad_devices:
    raise RuntimeError(
        f"Trainable parameters not on {device}: {bad_devices[:12]}"
        " — rerun Cells 4 and 5 in order, using the updated notebook.")
assert all(p.is_floating_point() for p in trainables)
print("LoRA placement verified:",len(lora_params),"tensors,",
      sum(p.numel() for p in lora_params),"parameters on",device)
param_groups=[]
if gain_params:param_groups.append({"params":gain_params,"lr":LR_GAIN_START,
                                    "weight_decay":0.0,"name":"CeNN gain"})
if lora_params:param_groups.append({"params":lora_params,"lr":LR_GAIN_START*LORA_LR_MULT,
                                    "weight_decay":0.0,"name":"all-layer LoRA"})
if core_params:param_groups.append({"params":core_params,"lr":LR_GAIN_START*CENN_CORE_LR_MULT,
                                    "weight_decay":1e-5,"name":"CeNN core"})
optimizer=torch.optim.AdamW(param_groups)
params=[p for g in optimizer.param_groups for p in g["params"]]
def gain_lr(step):
    progress=(step-1)/max(1,JOINT_STEPS-1)
    return LR_GAIN_END+0.5*(LR_GAIN_START-LR_GAIN_END)*(1+math.cos(math.pi*progress))
def make_snapshot():
    return [p.detach().cpu().contiguous().clone() for p in params]
@torch.no_grad()
def restore_snapshot(snapshot):
    assert len(params)==len(snapshot)
    for p,s in zip(params,snapshot):p.copy_(s.to(p.device,dtype=p.dtype))
def priority(metrics):
    # Always preserve the initial checkpoint if fine-tuning makes fidelity
    # worse; gold label quality is a secondary metric, JS is a tie breaker.
    return (round(metrics["teacher_agreement"],10),
            round(metrics["gold_accuracy"],10),-metrics["mean_js"])
best_metrics=BASELINE_DEV.copy()
best_state=make_snapshot()
best_step=0
history=[{"step":0,"learning_rate":None,"dev":BASELINE_DEV,"selected":True}]
order=list(range(len(train_records)))
rng=random.Random(SEED+737)
record_index=0
def configure_train():
    student_eval.eval()
    if GRAD_CHECKPOINTING:
        student_eval.encoder.train()
        for mod in student_eval.encoder.modules():
            if isinstance(mod,nn.Dropout):mod.eval()
checkpointing_active=False
if GRAD_CHECKPOINTING:
    try:
        student_eval.encoder.gradient_checkpointing_enable(
            gradient_checkpointing_kwargs={"use_reentrant":False})
        checkpointing_active=True
    except Exception as e:
        print("Gradient checkpointing unavailable:",str(e)[:180])
print("Trainable LoRA",sum(x.numel() for x in lora_params),
      "CeNN gain",len(gain_params),
      "CeNN core",sum(x.numel() for x in core_params))
# One real CUDA forward pass tests LoRA placement BEFORE expensive backprop.
assert train_records, "No training records available"
sample_batch={k:v.to(device,non_blocking=True)
              for k,v in train_records[0]["inputs"].items()}
with torch.inference_mode(),torch.autocast(
        "cuda",dtype=amp_dtype,enabled=FP16_AUTOCAST):
    smoke_logits,_=student_eval(**sample_batch)
assert torch.isfinite(smoke_logits).all(), "Nonfinite CUDA LoRA smoke logits"
print("CUDA LoRA/CeNN forward smoke PASS:",
      tuple(smoke_logits.shape),smoke_logits.device)
del smoke_logits,sample_batch
for step in range(1,JOINT_STEPS+1):
    rate=gain_lr(step)
    for group in optimizer.param_groups:
        scale=(1.0 if group["name"]=="CeNN gain" else
               LORA_LR_MULT if group["name"]=="all-layer LoRA"
               else CENN_CORE_LR_MULT)
        group["lr"]=rate*scale
    optimizer.zero_grad(set_to_none=True)
    configure_train()
    losses=[]
    for acc in range(GRAD_ACCUM):
        if record_index==0:rng.shuffle(order)
        rec=train_records[order[record_index]]
        record_index=(record_index+1)%len(order)
        batch={k:v.to(device,non_blocking=True) for k,v in rec["inputs"].items()}
        reference=rec["teacher"].to(device)
        gold=rec["gold"].to(device)
        with torch.autocast("cuda",dtype=amp_dtype,enabled=FP16_AUTOCAST):
            logits,_=student_eval(**batch)
            valid=batch["marker_mask"].bool()
            p=logits.float().masked_fill(~valid,-1e4)
            q=reference.float().masked_fill(~valid,-1e4)
            tau=KD_TEMPERATURE
            loss_kl=(tau*tau)*F.kl_div(
                F.log_softmax(p/tau,dim=-1),
                F.softmax(q/tau,dim=-1),reduction="batchmean")
            loss_ce=F.cross_entropy(p,gold)
            loss=KL_WEIGHT*loss_kl+GOLD_WEIGHT*loss_ce
        if not bool(torch.isfinite(loss).item()):
            raise FloatingPointError(f"Nonfinite loss at step {step}")
        scaler.scale(loss/GRAD_ACCUM).backward()
        losses.append(float(loss.detach()))
    scaler.unscale_(optimizer)
    norm=torch.nn.utils.clip_grad_norm_(params,MAX_GRAD_NORM,error_if_nonfinite=False)
    if not bool(torch.isfinite(norm).item()):
        optimizer.zero_grad(set_to_none=True)
        scaler.update()
        print("Skipped nonfinite-gradient step",step)
        continue
    scaler.step(optimizer)
    scaler.update()
    if step%EVAL_EVERY==0 or step==1 or step==JOINT_STEPS:
        metrics=eval_quality(student_eval,gate_records)
        improved=(priority(metrics)>priority(best_metrics) and
                  metrics["gold_accuracy"]>=BASELINE_DEV["gold_accuracy"]-0.005)
        if improved:
            best_metrics=metrics
            best_state=make_snapshot()
            best_step=step
        entry={"step":step,"gain_lr":rate,"lora_lr":rate*LORA_LR_MULT,
            "mean_loss":float(np.mean(losses)),"dev":metrics,
            "best_step":best_step}
        history.append(entry)
        (OUTPUT_DIR/"v7_progress.json").write_text(
            json.dumps({"baseline":BASELINE_DEV,"best":best_metrics,
                        "best_step":best_step,"history":history},indent=2))
        print(f"Step {step}/{JOINT_STEPS} gain_lr={rate:.6f} "
              f"agreement={metrics['teacher_agreement']:.4%} "
              f"gold={metrics['gold_accuracy']:.4%} "
              f"best_step={best_step}")
# This is the critical V6 fix: ALWAYS use best, even if that means step 0.
restore_snapshot(best_state)
student_eval.eval()
if checkpointing_active:
    try:student_eval.encoder.gradient_checkpointing_disable()
    except Exception:pass
FINAL_DEV_BEFORE_PRUNING=eval_quality(student_eval,gate_records)
assert priority(FINAL_DEV_BEFORE_PRUNING)>=priority(BASELINE_DEV)
assert ffn_norm_digest(student_eval)==BASELINE_FFN_HASH
assert abs(gain_lr(1)-LR_GAIN_START)<1e-12
assert abs(gain_lr(JOINT_STEPS)-LR_GAIN_END)<1e-12
# Merge every LoRA update into conventional QKV/WO weights: no runtime LoRA.
for layer in student_eval.encoder.layers:
    a=layer.attn
    if isinstance(a.Wqkv,MergeableLoRA):a.Wqkv=a.Wqkv.merge()
    if isinstance(a.Wo,MergeableLoRA):a.Wo=a.Wo.merge()
assert all(not isinstance(layer.attn.Wqkv,MergeableLoRA)
           for layer in student_eval.encoder.layers)
assert all(not isinstance(layer.attn.Wo,MergeableLoRA)
           for layer in student_eval.encoder.layers)
assert ffn_norm_digest(student_eval)==BASELINE_FFN_HASH
FINAL_DEV_MERGED=eval_quality(student_eval,gate_records)
print("Best optimizer step:",best_step,"FP16 eval:",FINAL_DEV_MERGED)
del optimizer,best_state
gc.collect();torch.cuda.empty_cache()


CUDA placement verified: 541 parameter tensors on cuda:0
LoRA placement verified: 112 tensors, 688128 parameters on cuda:0
Trainable LoRA 688128 CeNN gain 28 CeNN core 118329344
CUDA LoRA/CeNN forward smoke PASS: (2, 4) cuda:0
Step 1/250 gain_lr=0.010000 agreement=99.1667% gold=83.3333% best_step=0
Step 25/250 gain_lr=0.009818 agreement=99.1667% gold=83.3333% best_step=0
Skipped nonfinite-gradient step 35
Skipped nonfinite-gradient step 41
Step 50/250 gain_lr=0.009260 agreement=99.1667% gold=83.3333% best_step=0
Skipped nonfinite-gradient step 52
Step 75/250 gain_lr=0.008380 agreement=99.1667% gold=83.3333% best_step=0
Step 100/250 gain_lr=0.007265 agreement=99.1667% gold=83.3333% best_step=0
Step 125/250 gain_lr=0.006025 agreement=99.1667% gold=83.3333% best_step=0
Step 150/250 gain_lr=0.004783 agreement=99.1667% gold=83.3333% best_step=0
Step 175/250 gain_lr=0.003661 agreement=99.1667% gold=83.3333% best_step=0
Step 200/250 gain_lr=0.002770 agreement=99.1667% gold=83.3333% best_step=

In [6]:
#@title 6. Optional CeNN inference pruning across all 28 replaced attentions
# Allows fewer CeNN convolutions without restoring native attention.
# Small independent slice of DEV only; final full DEV recalculated afterwards.
pruned=[]
if AUTOPRUNE_CENN:
    mini=gate_records[:max(1,math.ceil(PRUNE_DECISIONS/BATCH))]
    before=eval_quality(student_eval,mini)
    nchecked=0
    for i,layer in enumerate(student_eval.encoder.layers):
        a=layer.attn
        if not a.use_cenn:continue
        if nchecked>=MAX_CENN_PRUNES:break
        nchecked+=1
        a.use_cenn=False
        after=eval_quality(student_eval,mini)
        # Disable expensive correction only if the mini DEV decisions are not worse.
        keep_off=(after["teacher_agreement"]>=before["teacher_agreement"] and
                  after["gold_accuracy"]>=before["gold_accuracy"] and
                  after["mean_js"]<=before["mean_js"]+1e-5)
        if keep_off:
            before=after
            pruned.append(i)
        else:
            a.use_cenn=True
    for entry in manifest_layers:
        entry["cenn_on"]=bool(student_eval.encoder.layers[entry["layer"]].attn.use_cenn)
FINAL_DEV_FP32=eval_quality(student_eval,gate_records)
# Mini-DEV pruning is tentative. Roll it back as a GROUP if the full
# independent DEV split is worse than the best pre-pruning checkpoint.
if priority(FINAL_DEV_FP32)<priority(FINAL_DEV_MERGED):
    for idx in pruned:
        student_eval.encoder.layers[idx].attn.use_cenn=True
    pruned=[]
    for entry in manifest_layers:
        entry["cenn_on"]=bool(student_eval.encoder.layers[entry["layer"]].attn.use_cenn)
    FINAL_DEV_FP32=eval_quality(student_eval,gate_records)
    print("Rolled back CeNN pruning because full DEV quality decreased.")
print("CeNN branches pruned (all 28 attention replacements REMAIN):",pruned)
print("Full DEV after pruning",FINAL_DEV_FP32)
assert ffn_norm_digest(student_eval)==BASELINE_FFN_HASH


CeNN branches pruned (all 28 attention replacements REMAIN): [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27]
Full DEV after pruning {'n': 240, 'teacher_agreement': 0.9916666666666667, 'gold_accuracy': 0.8333333333333334, 'teacher_gold_accuracy': 0.8416666666666667, 'mean_js': 0.00010349645356958112}


In [7]:
#@title 7. Full-model FP16 export validation, sealed HOLDOUT + fair FP16 benchmarks
# Actual FP16 weights in GPU memory; cache teacher logits before CPU release.
teacher.to(device).eval()
holdout_items=make_items_from_rows(holdout_rows,SEED+999,False)[:HOLDOUT_DECISIONS]
assert len(holdout_items)==HOLDOUT_DECISIONS
holdout_records=bucket_records(holdout_items)
# Same dtype for timing teacher and substitute.
teacher.half()
student_eval.half()
student_eval.eval().requires_grad_(False)
teacher.eval().requires_grad_(False)
# FP16 conversion does NOT guarantee a changed SHA256 digest:
# ffn_norm_digest intentionally hashes float32-casted values, and weights
# representable exactly in float16 keep the same value/hash.
# Verify the correct properties instead:
assert all(p.dtype==torch.float16 for p in student_eval.parameters()), (
    "Some student parameters were not converted to FP16")
assert all(p.dtype==torch.float16 for p in teacher.parameters()), (
    "Some teacher parameters were not converted to FP16")
for index,(student_layer,teacher_layer) in enumerate(
        zip(student_eval.encoder.layers,teacher.encoder.layers)):
    for name in ("mlp","mlp_norm","attn_norm"):
        student_state=getattr(student_layer,name).state_dict()
        teacher_state=getattr(teacher_layer,name).state_dict()
        assert student_state.keys()==teacher_state.keys(),(index,name)
        for key,value in student_state.items():
            reference=teacher_state[key]
            assert value.dtype==reference.dtype,(index,name,key,"dtype")
            if value.is_floating_point():
                assert value.dtype==torch.float16,(index,name,key,"not-fp16")
            assert torch.equal(value,reference),(
                "Frozen FFN/normalization differs from FP16 teacher",
                index,name,key)
# This equality is a useful checksum; inequality with the original FP32
# digest is NOT required and is generally unsafe to assert.
FP16_FFN_HASH=ffn_norm_digest(student_eval)
assert FP16_FFN_HASH==ffn_norm_digest(teacher)
print("PASS: FP16 tensors and all 28 frozen FFNs/norms exactly match teacher")
print("Native FP32 fingerprint:",BASELINE_FFN_HASH)
print("Native FP16 fingerprint:",FP16_FFN_HASH)
FP16_DEV=eval_quality(student_eval,gate_records)
FP16_HOLDOUT=eval_quality(student_eval,holdout_records)
print("True FP16 inference DEV:",FP16_DEV)
print("True FP16 inference sealed holdout:",FP16_HOLDOUT)
@torch.inference_mode()
def benchmark(model,inp,repeats=BENCH_REPEATS):
    # Same FP16 inference mode for BOTH teacher and substitute.
    # The auxiliary action head constructs float32 statistics. The patched
    # StandaloneDecisionModel casts those to the FP16 head dtype; autocast
    # also keeps older active Colab kernels safe without retraining.
    model.eval()
    def run_one():
        with torch.autocast("cuda",dtype=torch.float16,enabled=FP16_AUTOCAST):
            return model(**inp)
    for _ in range(4):
        logits,_=run_one()
        if not torch.isfinite(logits).all():
            raise FloatingPointError("Nonfinite FP16 benchmark logits")
    torch.cuda.synchronize()
    times=[]
    for _ in range(repeats):
        start=torch.cuda.Event(enable_timing=True)
        end=torch.cuda.Event(enable_timing=True)
        start.record()
        run_one()
        end.record()
        end.synchronize()
        times.append(float(start.elapsed_time(end)))
    return {"median_ms":float(np.median(times)),
            "p95_ms":float(np.percentile(times,95)),
            "repeats":len(times),"precision":"fp16_weights_fp16_autocast"}
short=[x for x in gate_items+probe_items if len(x["ids"])<=160]
long=[x for x in gate_items+probe_items if 384<=len(x["ids"])<=512]
speed=[]
for label,pool in (("short",short),("long",long)):
    for n in (1,2):
        if len(pool)<n:
            speed.append({"class":label,"batch":n,"status":"no_real_samples"})
            continue
        inp=direct_batch(pool[:n])
        a=benchmark(teacher,inp)
        b=benchmark(student_eval,inp)
        speed.append({"class":label,"batch":n,
                      "lengths":[len(x["ids"]) for x in pool[:n]],
                      "teacher_fp16":a,"student_fp16":b,
                      "speedup":a["median_ms"]/max(1e-9,b["median_ms"]),
                      "status":"measured"})
        print(label,"B",n,"speedup",speed[-1]["speedup"])
# Keep inference student FP16 in GPU memory; teacher can be moved to CPU.
teacher.cpu();gc.collect();torch.cuda.empty_cache()


PASS: FP16 tensors and all 28 frozen FFNs/norms exactly match teacher
Native FP32 fingerprint: 766166b2aa6376e9023fff6e2b63a2cb80618f2065597342f5cafe8be57670e6
Native FP16 fingerprint: 766166b2aa6376e9023fff6e2b63a2cb80618f2065597342f5cafe8be57670e6
True FP16 inference DEV: {'n': 240, 'teacher_agreement': 0.9916666666666667, 'gold_accuracy': 0.8333333333333334, 'teacher_gold_accuracy': 0.8416666666666667, 'mean_js': 0.00010393189521778664}
True FP16 inference sealed holdout: {'n': 500, 'teacher_agreement': 1.0, 'gold_accuracy': 0.864, 'teacher_gold_accuracy': 0.864, 'mean_js': 1.310033980814751e-05}
short B 1 speedup 0.8348016355224519
short B 2 speedup 0.793604019538434
long B 1 speedup 0.7761730651864711
long B 2 speedup 0.970530647488806


In [8]:
#@title 8. Complete standalone FP16 checkpoint + metadata (never FP32-upcast)
student_eval.cpu().eval()
model_file=EXPORT_DIR/"model.safetensors"
save_model(student_eval,str(model_file))
assert model_file.stat().st_size>0
tokenizer.save_pretrained(str(EXPORT_DIR/"tokenizer"))
(EXPORT_DIR/"encoder").mkdir(exist_ok=True)
shutil.copy2(source_dir/"encoder"/"config.json",
             EXPORT_DIR/"encoder"/"config.json")
shutil.copy2(source_dir/"rl_agent_config.json",
             EXPORT_DIR/"rl_agent_config.json")
shutil.copy2(Path(fp16_loader.__file__),EXPORT_DIR/"all28_fp16.py")
# Keep conventional all28_hf.py optional for tooling compatibility.
manifest={
  "architecture":"jev_cenn_all28_fp16_stable_v7",
  "source_model":SOURCE_MODEL,"initialized_from":init_method,
  "model_dtype":"float16","fp16_training_autocast":FP16_AUTOCAST,
  "original_ffn_norm_sha256_fp32":BASELINE_FFN_HASH,
  "original_ffn_norm_sha256_fp16":ffn_norm_digest(student_eval),
  "ffn_norm_unchanged_except_fp16_rounding":True,
  "all_28_attention_replaced":True,
  "attention_layers":manifest_layers,
  "train_steps":JOINT_STEPS,"selected_best_step":best_step,
  "gain_lr_start":LR_GAIN_START,"gain_lr_end":LR_GAIN_END,
  "lora_rank":LORA_RANK,"lora_merge_complete":True,
  "dev_initial":BASELINE_DEV,"dev_fp16":FP16_DEV,
  "holdout_fp16":FP16_HOLDOUT,"paired_fp16_speed":speed,
  "pruned_cenn_layers":pruned,
  "model_parameter_count":sum(p.numel() for p in student_eval.parameters()),
  "created_utc":datetime.now(timezone.utc).isoformat(),
}
(EXPORT_DIR/"all28_manifest.json").write_text(
    json.dumps(manifest,indent=2,allow_nan=False),encoding="utf-8")
(EXPORT_DIR/"requirements.txt").write_text(
   "torch>=2.4\ntransformers==4.57.6\n"
   "safetensors>=0.4\nhuggingface_hub>=0.25,<1\n"
   "git+https://github.com/vtavakkoli/TinyCeNN-LM.git@645723441fb81c62d59231bffd29a177a5edb3ac\n",
   encoding="utf-8")
readme=f"""---
library_name: pytorch
tags:
- cenn
- modernbert
- fp16
- attention-substitution
---

# Jev-CeNN All-28 Stable FP16 V7

Full **FP16** typed-decision model. All **28 attention modules** replaced;
original pretrained FFNs/norms preserved, except unavoidable FP16 rounding.
LoRA attention fine-tuning is **merged** into original-shaped QKV/WO weights
before export; no additional inference LoRA matmul.

- Initialization: {init_method}
- Training learning rate CeNN gains: {LR_GAIN_START} → {LR_GAIN_END}
- Best checkpoint (DEV-selected): step {best_step}
- FP16 DEV teacher agreement: {FP16_DEV['teacher_agreement']:.4f}
- FP16 sealed holdout agreement: {FP16_HOLDOUT['teacher_agreement']:.4f}
- FP16 sealed holdout gold accuracy: {FP16_HOLDOUT['gold_accuracy']:.4f}

## Installation
```bash
pip install "transformers==4.57.6" safetensors huggingface_hub
pip install "git+https://github.com/vtavakkoli/TinyCeNN-LM.git@645723441fb81c62d59231bffd29a177a5edb3ac"
```

## Load (the supplied loader preserves FP16 on CUDA)
```python
from tinycenn_lm.all28_fp16 import load_all28
runtime=load_all28("REPLACE_WITH_HF_REPO",device="cuda")
print(runtime.decide(
    {{"road_direction":"left","speed":"7"}},
    ["left","straight","right"]))
```

The checkpoint is a typed-decision model, not a generative chat model.
Use `tinycenn_lm.all28_fp16.load_all28` rather than
`AutoModel.from_pretrained`. All model weights, tokenizer/config, and
architecture loader are included; no original checkpoint is required.

The native model is from `{SOURCE_MODEL}`. Check upstream license
and redistribution terms before changing the repository to public.
"""
(EXPORT_DIR/"README.md").write_text(readme,encoding="utf-8")
(OUTPUT_DIR/"v7_final_report.json").write_text(json.dumps(
    {"manifest":manifest,"history":history},indent=2,allow_nan=False),
    encoding="utf-8")
print("Exported true FP16:",model_file,
      "size GiB:",round(model_file.stat().st_size/1024**3,3))


Exported true FP16: /content/Jev_CeNN_All28_FP16_Stable_V7/huggingface_model/model.safetensors size GiB: 1.005


In [9]:
#@title 9. Strict reload parity: FP16 must remain FP16 and FFNs intact
# The loader will instantiate FP16 weights BEFORE strict safetensors loading.
runtime=build_all28_from_dir(EXPORT_DIR,device="cuda")
loaded=runtime.model
assert len(loaded.encoder.layers)==28
assert all(isinstance(layer.attn,WindowCeNNAttention)
           for layer in loaded.encoder.layers)
assert all(p.dtype==torch.float16 for p in loaded.parameters())
assert ffn_norm_digest(loaded)==ffn_norm_digest(student_eval)
probe=tokenizer("Should the controller turn left or continue straight?",
                return_tensors="pt",max_length=64,truncation=True)
assert probe["input_ids"].shape[1]>1
inp={"input_ids":probe["input_ids"].to(device),
     "attention_mask":probe["attention_mask"].to(device),
     "marker_pos":torch.tensor([[0,1]],device=device),
     "marker_mask":torch.ones((1,2),dtype=torch.bool,device=device),
     "qtype":torch.tensor([0],device=device)}
student_eval.to(device).eval()
# Both exported and in-memory models use identical FP16 compute settings.
# Avoid the Float-vs-Half auxiliary action-head mismatch in older Colab
# sessions whose Python package was imported before the GitHub dtype fix.
with torch.inference_mode(),torch.autocast(
        "cuda",dtype=torch.float16,enabled=FP16_AUTOCAST):
    x,_=student_eval(**inp)
    y,_=loaded(**inp)
    error=float((x.float()-y.float()).abs().max())
assert error<=0.02,f"FP16 serialization parity failure: max logit abs {error}"
print("FP16 strict reload PASSED: max abs logit delta",error)
del loaded,runtime
student_eval.cpu();gc.collect();torch.cuda.empty_cache()


FP16 strict reload PASSED: max abs logit delta 0.0


In [10]:
#@title 10. Package ALL V7 results + FULL standalone FP16 model into ZIPs, download both
# This cell runs BEFORE optional Hugging Face upload. A Hub error must not
# prevent collecting the model and experiment results on your own computer.
from zipfile import ZipFile, ZIP_DEFLATED, ZIP_STORED
from IPython.display import display, FileLink

RESULTS_ZIP=OUTPUT_DIR/"Jev_CeNN_All28_V7_FP16_Results.zip"
MODEL_ZIP=OUTPUT_DIR/"Jev_CeNN_All28_V7_FP16_Standalone_Model.zip"

# Save a concise machine-readable summary, in addition to the complete
# manifest, progress JSON and final report already written by earlier cells.
results_summary={
    "architecture":"jev_cenn_all28_fp16_stable_v7",
    "model_dtype":"float16",
    "attention_layers_replaced":28,
    "original_ffns_unchanged_except_fp16_rounding":True,
    "best_step":int(best_step),
    "dev_teacher_agreement":float(FP16_DEV["teacher_agreement"]),
    "holdout_teacher_agreement":float(FP16_HOLDOUT["teacher_agreement"]),
    "holdout_gold_accuracy":float(FP16_HOLDOUT["gold_accuracy"]),
    "teacher_holdout_gold_accuracy":float(FP16_HOLDOUT["teacher_gold_accuracy"]),
    "latency":speed,
    "ffn_sha256_fp16":FP16_FFN_HASH,
    "model_safetensors_bytes":int(model_file.stat().st_size),
}
(OUTPUT_DIR/"v7_results_summary.json").write_text(
    json.dumps(results_summary,indent=2,allow_nan=False),encoding="utf-8")

# Lightweight archive: results and logs, small enough for reliable downloads.
result_files=sorted(
    [p for p in OUTPUT_DIR.glob("*.json") if p.is_file()]
)
assert {"v7_final_report.json","v7_results_summary.json"}.issubset(
    {p.name for p in result_files})
with ZipFile(RESULTS_ZIP,"w") as z:
    for p in result_files:
        z.write(p,arcname="results/"+p.name,compress_type=ZIP_DEFLATED)

# Complete deployable model: all FP16 safetensors, tokenizer/config, loader,
# and a copy of the evaluation/training reports.
# FP16 safetensors are stored without recompression to save Colab CPU time.
model_files=sorted(p for p in EXPORT_DIR.rglob("*") if p.is_file())
assert (EXPORT_DIR/"model.safetensors") in model_files
assert (EXPORT_DIR/"all28_manifest.json") in model_files
assert (EXPORT_DIR/"all28_fp16.py") in model_files
with ZipFile(MODEL_ZIP,"w",allowZip64=True) as z:
    for p in model_files:
        compression=ZIP_STORED if p.name=="model.safetensors" else ZIP_DEFLATED
        z.write(p,arcname="model/"+p.relative_to(EXPORT_DIR).as_posix(),
                compress_type=compression)
    for p in result_files:
        z.write(p,arcname="results/"+p.name,compress_type=ZIP_DEFLATED)

for archive_path in (RESULTS_ZIP,MODEL_ZIP):
    with ZipFile(archive_path) as z:
        error=z.testzip()
        assert error is None, f"Corrupt ZIP entry: {error}"
        print(archive_path.name, "files",len(z.namelist()),
              "size MiB",round(archive_path.stat().st_size/2**20,1))
print("Results:",RESULTS_ZIP)
print("Complete model:",MODEL_ZIP)
display(FileLink(str(RESULTS_ZIP)))
display(FileLink(str(MODEL_ZIP)))

# Colab browser downloads require an active UI session. They may be blocked
# by a browser's multi-download policy, particularly for large model ZIPs.
try:
    from google.colab import files
    if AUTO_DOWNLOAD_RESULTS_ZIP:
        files.download(str(RESULTS_ZIP))
    if AUTO_DOWNLOAD_MODEL_ZIP:
        files.download(str(MODEL_ZIP))
except (ImportError,AttributeError,RuntimeError) as exc:
    print("Automatic browser download unavailable:",str(exc)[:240])
    print("Use the links above or Colab's Files sidebar to download each ZIP.")


Jev_CeNN_All28_V7_FP16_Results.zip files 3 size MiB 0.0
Jev_CeNN_All28_V7_FP16_Standalone_Model.zip files 13 size MiB 1029.9
Results: /content/Jev_CeNN_All28_FP16_Stable_V7/Jev_CeNN_All28_V7_FP16_Results.zip
Complete model: /content/Jev_CeNN_All28_FP16_Stable_V7/Jev_CeNN_All28_V7_FP16_Standalone_Model.zip


/content/Jev_CeNN_All28_FP16_Stable_V7/Jev_CeNN_All28_V7_FP16_Results.zip

/content/Jev_CeNN_All28_FP16_Stable_V7/Jev_CeNN_All28_V7_FP16_Standalone_Model.zip

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [11]:
#@title 11. Publish V7 complete FP16 model on Hugging Face (private by default)
if UPLOAD_TO_HF:
    token=get_token()
    if not token:
        login()
        token=get_token()
    if not token:raise RuntimeError("Hugging Face write token required")
    api=HfApi(token=token)
    username=api.whoami(token=token)["name"]
    repo_id=f"{username}/{HF_REPO_NAME}"
    card=EXPORT_DIR/"README.md"
    card.write_text(card.read_text(encoding="utf-8").replace(
       "REPLACE_WITH_HF_REPO",repo_id),encoding="utf-8")
    api.create_repo(repo_id=repo_id,repo_type="model",
                    private=HF_PRIVATE,exist_ok=True)
    api.upload_folder(repo_id=repo_id,repo_type="model",
                      folder_path=str(EXPORT_DIR),
                      commit_message="New all-28 attention frozen-FFN FP16 model, stable KD and merged LoRA")
    remote=set(api.list_repo_files(repo_id=repo_id,repo_type="model"))
    required={"model.safetensors","all28_fp16.py",
              "all28_manifest.json","encoder/config.json",
              "rl_agent_config.json","README.md"}
    assert required.issubset(remote),sorted(required-remote)
    print("Published:",f"https://huggingface.co/{repo_id}")
else:
    print("Export is ready locally:",EXPORT_DIR)


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...e_model/model.safetensors:   0%|          |  525kB / 1.08GB            

Published: https://huggingface.co/vtava/jev-cenn-all28-fp16-stable-v7
